<a href="https://colab.research.google.com/github/bernardlawes/Colab-Roboflow/blob/main/Roboflow_Ship_Cargo_2_Stage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Inference SDK

In [ ]:
!pip install inference-sdk
# !pip install roboflow inference-sdk

# Select Input

In [ ]:
IMAGE_URL = "https://dam.krohne.com/t_ar43_cr_c/e_trim:0/w_auto/q_auto/dpr_auto/f_auto/d_im-other:image-not-available.png/im-contract-photography/orange-black-loaded-container-ship-harbour.jpg"

# Run Inference

In [ ]:
from inference_sdk import InferenceHTTPClient

# 🔹 Step 1: Initialize Roboflow API Client
client = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key=userdata.get('ROBOFLOW_API_KEY')
)


# 🔹 Step 2: Send Image for Inference
result = client.run_workflow(
    workspace_name="robo-hello-world",
    workflow_id="detect-count-and-visualize-3",
    images={
        "image": IMAGE_URL
    },
    use_cache=True # cache workflow definition for 15 minutes
)

# View JSON Result (Pretty Format)

In [ ]:
import json
# 🔹 Step 3: Display Full JSON (For Debugging)
print(json.dumps(result, indent=4))  # Pretty-print JSON

# Process / Parse JSON Result

In [ ]:
# 🔹 Step 3: Access First Item (Since it's a list)
data = result[0]

# 🔹 Step 4: Extract Overall Image Info
image_info = data["predictions"]["image"]
image_width = image_info["width"]
image_height = image_info["height"]

# 🔹 Step 5: Print Image Size
print(f"Image Size: {image_width}x{image_height}")

print("\n")


In [ ]:

# 🔹 Step 6: Extract Predictions (List of Detected Objects)
predictions = data["predictions"]["predictions"]

# 🔹 Step 7: Loop Through Detected Objects and Print Information
for obj in predictions:
    class_name = obj["class"]
    confidence = obj["confidence"] * 100  # Convert to percentage
    x, y = obj["x"], obj["y"]  # Center coordinates
    width, height = obj["width"], obj["height"]  # Bounding box size

    print(f"Detected {class_name}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Location: x={x}, y={y}")
    print(f"Bounding Box: Width={width}, Height={height}")
    print(f"Detection ID: {obj['detection_id']}")

    print("\n")


# Read in the image from URL

In [ ]:
import cv2
import numpy as np
import requests
from google.colab.patches import cv2_imshow

image_np = np.asarray(bytearray(requests.get(IMAGE_URL).content), dtype=np.uint8)
image = cv2.imdecode(image_np, cv2.IMREAD_COLOR)

# Create a copy of the image that I will use to draw on
image_canvas = image.copy()

# Display Detected Ships

In [ ]:
# Step 4: Draw bounding boxes around detected objects
for obj in predictions:
    x, y, width, height = int(obj["x"]), int(obj["y"]), int(obj["width"]), int(obj["height"])
    class_name = obj["class"]
    confidence = obj["confidence"] * 100  # Convert to percentage

    # Define bounding box color (green) and thickness
    color = (0, 255, 0)
    thickness = 2

    # Draw rectangle around detected object
    top_left = (x - width // 2, y - height // 2)
    bottom_right = (x + width // 2, y + height // 2)
    cv2.rectangle(image_canvas, top_left, bottom_right, color, thickness)

    # Put class name and confidence above bounding box
    label = f"{class_name}: {confidence:.2f}%"
    cv2.putText(image_canvas, label, (top_left[0], top_left[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# Step 5: Display the result
cv2_imshow(image_canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
  # Calculate top-left and bottom-right coordinates
  x1 = max(0, x - width // 2)
  y1 = max(0, y - height // 2)
  x2 = min(image.shape[1], x + width // 2)
  y2 = min(image.shape[0], y + height // 2)

  # Crop the image
  cropped_image = image[y1-1:y2+1, x1+1:x2-1]

  # 🔹 Step 5: Display the result
  cv2_imshow(cropped_image)
  cv2.waitKey(0)
  cv2.destroyAllWindows()